In [ ]:
import pandas as pd

df = pd.read_csv("cgv_final_integrated_v2.csv", encoding="utf-8-sig")

print("Số dòng:", len(df))
print("Số cột:", len(df.columns))

df.head()

In [ ]:
# Tạo embedding model
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
# Kết nối Qdrant Cloud
import os
from qdrant_client import QdrantClient

QDRANT_URL = "https://101c42db-7067-49f4-a9a6-9ae7934bab1c.eu-central-1-0.aws.cloud.qdrant.io"
# Không ghi API key trực tiếp vào notebook: đặt biến môi trường QDRANT_API_KEY
# hoặc dán key vào chuỗi bên dưới khi chạy (đừng chia sẻ/commit file có key).
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY", "DAN_API_KEY_CUA_BAN_VAO_DAY")

client = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY
)

print("Kết nối Qdrant thành công!")

In [ ]:
# Tạo Collection
from qdrant_client.models import Distance, VectorParams

collection_name = "cgv_movies_v2"

client.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(
        size=384,
        distance=Distance.COSINE
    )
)

print("Đã tạo collection:", collection_name)

In [ ]:
# Tạo embedding cho GeneratedText
texts = df["GeneratedText"].fillna("").astype(str).tolist()

embeddings = model.encode(
    texts,
    show_progress_bar=True
)

print("Số vector:", len(embeddings))
print("Số chiều:", len(embeddings[0]))

In [ ]:
# Đưa dữ liệu vào Qdrant
from qdrant_client.models import PointStruct

def to_int(value):
    """Chuyển về int, trả về None nếu thiếu dữ liệu."""
    return int(value) if pd.notna(value) else None

points = []

for i, row in df.iterrows():
    points.append(
        PointStruct(
            id=i + 1,
            vector=embeddings[i].tolist(),
            payload={
                "MovieName": row["MovieName"],
                "Cinema": row["Cinema"],
                "ShowDate": str(row["ShowDate"]),
                "Format": row["Format"],
                "ShowTime": row["ShowTime"],
                "GeneratedText": row["GeneratedText"],
                "TicketPrice": to_int(row["TicketPrice"]),
                "AvailableSeats": to_int(row["AvailableSeats"]),
                "TotalSeats": to_int(row["TotalSeats"]),
                "MovieURL": row["MovieURL"],
                "Rating": row["Rating"],
                "Director": row["Director"],
                "Actors": row["Actors"],
                "Genre": row["Genre"],
                "ReleaseDate": str(row["ReleaseDate"]),
                "Duration": row["Duration"],
                "Language": row["Language"]
            }
        )
    )

print("Đã tạo", len(points), "points")

In [ ]:
# Upload
client.upsert(
    collection_name=collection_name,
    points=points,
    wait=True
)

print("Đã upload dữ liệu vào Qdrant!")

In [ ]:
info = client.get_collection(collection_name)

print(info)

***
# Test tìm kiếm ngữ nghĩa

In [ ]:
# Tạo vector cho câu hỏi
query = "phim Avengers"

query_vector = model.encode(query).tolist()

In [ ]:
# Tìm trong Qdrant
results = client.query_points(
    collection_name=collection_name,
    query=query_vector,
    limit=5
).points

In [ ]:
# Hiển thị
for result in results:
    print("Score:", result.score)
    print("Movie:", result.payload["MovieName"])
    print("Cinema:", result.payload["Cinema"])
    print("ShowTime:", result.payload["ShowTime"])
    print("TicketPrice:", result.payload["TicketPrice"])
    print("AvailableSeats:", result.payload["AvailableSeats"], "/", result.payload["TotalSeats"])
    print("-" * 50)